# EDA — Oportunidades em meio ao Caos

Análise exploratória da camada `processed/mensal` consumida diretamente do MinIO via API S3 (`boto3`).

**Objetivo:** entender estrutura, cobertura temporal, missing values, estatísticas, comportamento das séries e correlações antes da etapa de Machine Learning.

> Este notebook é somente de análise. Ele não altera os arquivos do MinIO.

## 1. Configuração

As credenciais devem ficar no `.env` do projeto, nunca diretamente no notebook.

In [2]:
import sys
import site

print("Python:")
print(sys.executable)

print("\nSite-packages:")
print(site.getsitepackages())

print("\nBoto3:")
try:
    import boto3
    print("OK")
    print(boto3.__version__)
    print(boto3.__file__)
except Exception as e:
    print("ERRO:", e)


Python:
c:\Users\Fabio\Desktop\HandsOn\handson_mack_2sem\mvp_resiliencia_dados\.venv\Scripts\python.exe

Site-packages:
['c:\\Users\\Fabio\\Desktop\\HandsOn\\handson_mack_2sem\\mvp_resiliencia_dados\\.venv', 'c:\\Users\\Fabio\\Desktop\\HandsOn\\handson_mack_2sem\\mvp_resiliencia_dados\\.venv\\Lib\\site-packages']

Boto3:
OK
1.43.80
c:\Users\Fabio\Desktop\HandsOn\handson_mack_2sem\mvp_resiliencia_dados\.venv\Lib\site-packages\boto3\__init__.py


In [7]:
import os
from io import BytesIO
from pathlib import Path

import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dotenv import load_dotenv

load_dotenv()

MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
MINIO_SECURE = os.getenv("MINIO_SECURE", "false").lower() == "true"

BUCKET_RAW = os.getenv("MINIO_BUCKET_RAW", "raw")
BUCKET_PROCESSED = os.getenv("MINIO_BUCKET_PROCESSED", "processed")
BUCKET_CURATED = os.getenv("MINIO_BUCKET_CURATED", "curated")

if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
    raise ValueError(
        "Credenciais do MinIO não encontradas. "
        "Configure MINIO_ROOT_USER e MINIO_ROOT_PASSWORD no .env."
    )

print("Credenciais encontradas!")
print("Endpoint:", MINIO_ENDPOINT)
print("Secure:", MINIO_SECURE)
print("Bucket RAW:", BUCKET_RAW)
print("Bucket PROCESSED:", BUCKET_PROCESSED)
print("Bucket CURATED:", BUCKET_CURATED)

Credenciais encontradas!
Endpoint: localhost:9000
Secure: False
Bucket RAW: raw
Bucket PROCESSED: processed
Bucket CURATED: curated


## 2. Conexão com o MinIO

In [9]:
s3 = boto3.client(
    "s3",
    endpoint_url=f"http{'s' if MINIO_SECURE else ''}://{MINIO_ENDPOINT}",
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1"
)

response = s3.list_buckets()

print("Conexão com MinIO OK!")
print("Buckets encontrados:")

for bucket in response["Buckets"]:
    print("-", bucket["Name"])

Conexão com MinIO OK!
Buckets encontrados:
- curated
- processed
- raw


## 3. Inventário dos objetos processados

Primeiro vamos descobrir o que realmente existe no prefixo `processed/mensal/`.

In [ ]:
PREFIX = "processed/mensal/"

objects = []
paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=MINIO_BUCKET, Prefix=PREFIX):
    objects.extend(page.get("Contents", []))

if not objects:
    print(f"Nenhum objeto encontrado em {PREFIX}")
else:
    inventory = pd.DataFrame([
        {
            "key": obj["Key"],
            "size_kb": round(obj["Size"] / 1024, 2),
            "last_modified": obj["LastModified"],
        }
        for obj in objects
    ])
    display(inventory)


## 4. Localizar a base mensal

O código abaixo aceita tanto `.parquet` quanto `.csv`, permitindo que o pipeline evolua sem precisar reescrever o notebook.

In [ ]:
if not objects:
    raise FileNotFoundError("Não há arquivos em processed/mensal/ no MinIO.")

keys = [obj["Key"] for obj in objects]
data_keys = [
    key for key in keys
    if key.lower().endswith((".parquet", ".csv"))
]

if not data_keys:
    raise FileNotFoundError("Nenhum CSV ou Parquet encontrado em processed/mensal/.")

print("Arquivos de dados encontrados:")
for key in data_keys:
    print(f" - {key}")

# Se houver mais de um arquivo, altere DATA_KEY manualmente.
DATA_KEY = data_keys[0]
print(f"\nArquivo selecionado: {DATA_KEY}")

In [ ]:
obj = s3.get_object(Bucket=MINIO_BUCKET, Key=DATA_KEY)
raw_bytes = obj["Body"].read()

if DATA_KEY.lower().endswith(".parquet"):
    df = pd.read_parquet(BytesIO(raw_bytes))
elif DATA_KEY.lower().endswith(".csv"):
    df = pd.read_csv(BytesIO(raw_bytes))
else:
    raise ValueError("Formato não suportado.")

print(f"Linhas:   {df.shape[0]:,}")
print(f"Colunas:  {df.shape[1]}")

display(df.head())

## 5. Padronização mínima para análise

A EDA não deve modificar o arquivo no MinIO. Aqui apenas criamos uma cópia em memória para análise.

In [ ]:
df_eda = df.copy()

if "data" not in df_eda.columns:
    raise ValueError(
        "A base processada deve possuir a coluna 'data'. "
        f"Colunas encontradas: {df_eda.columns.tolist()}"
    )

df_eda["data"] = pd.to_datetime(df_eda["data"], errors="coerce")
df_eda = df_eda.sort_values("data").reset_index(drop=True)

print("Período:")
print(f"Início: {df_eda['data'].min()}")
print(f"Fim:    {df_eda['data'].max()}")
print(f"Meses:  {df_eda['data'].nunique():,}")


## 6. Estrutura e tipos

In [ ]:
display(pd.DataFrame({
    "coluna": df_eda.columns,
    "tipo": [df_eda[c].dtype for c in df_eda.columns],
    "na": [df_eda[c].isna().sum() for c in df_eda.columns],
    "na_pct": [round(df_eda[c].isna().mean() * 100, 2) for c in df_eda.columns],
    "unicos": [df_eda[c].nunique(dropna=True) for c in df_eda.columns],
})).sort_values("na_pct", ascending=False)

## 7. Missing values

Nesta etapa não vamos preencher automaticamente os valores ausentes. O objetivo é identificar onde eles estão e decidir o tratamento de acordo com a natureza econômica de cada variável.

In [ ]:
missing = (
    df_eda.isna()
    .sum()
    .rename("missing")
    .to_frame()
)
missing["pct"] = (missing["missing"] / len(df_eda) * 100).round(2)
display(missing.sort_values("missing", ascending=False))

## 8. Estatísticas descritivas

A análise considera somente colunas numéricas.

In [ ]:
numeric_cols = df_eda.select_dtypes(include=np.number).columns.tolist()
display(df_eda[numeric_cols].describe().T)

## 9. Evolução temporal das séries

A primeira leitura deve ser visual: tendência, mudanças de nível, períodos de maior variação e possíveis quebras.

In [ ]:
series_principais = [
    c for c in [
        "ibc_br", "selic", "usd_brl", "ipca", "pib",
        "desemprego", "ibovespa", "icon", "iee", "ifnc"
    ]
    if c in df_eda.columns
]

for col in series_principais:
    plt.figure(figsize=(12, 4))
    plt.plot(df_eda["data"], df_eda[col])
    plt.title(col)
    plt.xlabel("Data")
    plt.ylabel(col)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Retornos / variações mensais

Para séries de nível, a variação percentual mensal ajuda a comparar movimentos ao longo do tempo.

In [ ]:
cols_retorno = [
    c for c in ["ibovespa", "icon", "iee", "ifnc", "usd_brl", "ibc_br"]
    if c in df_eda.columns
]

retornos = pd.DataFrame(index=df_eda.index)
retornos["data"] = df_eda["data"]

for col in cols_retorno:
    retornos[f"{col}_var_mom"] = df_eda[col].pct_change()

display(retornos.head(12))

## 11. Variação em 3 e 12 meses

Essas variáveis são úteis para identificar mudanças de tendência e movimentos persistentes, sem ainda constituírem um modelo de Machine Learning.

In [ ]:
janelas = [3, 12]
variacoes = pd.DataFrame({"data": df_eda["data"]})

for col in cols_retorno:
    for janela in janelas:
        variacoes[f"{col}_var_{janela}m"] = df_eda[col].pct_change(janela)

display(variacoes.head(15))

## 12. Correlação

A correlação é exploratória. Ela não implica causalidade e será usada apenas para levantar hipóteses para a etapa seguinte.

In [ ]:
corr = df_eda[numeric_cols].corr()

plt.figure(figsize=(12, 9))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlação")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Matriz de correlação")
plt.tight_layout()
plt.show()

## 13. Correlação dos retornos dos índices B3

Para comparar resiliência entre os índices, faz mais sentido observar também seus movimentos relativos, e não somente seus níveis.

In [ ]:
b3_cols = [c for c in ["ibovespa", "icon", "iee", "ifnc"] if c in df_eda.columns]

if b3_cols:
    b3_returns = df_eda[b3_cols].pct_change()
    display(b3_returns.corr())
else:
    print("Nenhuma série B3 encontrada com os nomes esperados.")

## 14. Verificação de duplicidade temporal

In [ ]:
duplicadas = df_eda[df_eda["data"].duplicated(keep=False)]

print(f"Datas duplicadas: {duplicadas['data'].nunique()}")
display(duplicadas.head(20))

## 15. Cobertura mensal

Verifica se a base possui todos os meses esperados dentro do período.

In [ ]:
datas = pd.to_datetime(df_eda["data"]).dt.to_period("M")
esperadas = pd.period_range(datas.min(), datas.max(), freq="M")
presentes = pd.Index(datas.unique())
faltantes = esperadas.difference(presentes)

print(f"Meses esperados: {len(esperadas)}")
print(f"Meses presentes: {len(presentes)}")
print(f"Meses ausentes:  {len(faltantes)}")

if len(faltantes) > 0:
    print("\nPrimeiros meses ausentes:")
    print(list(faltantes[:20]))

## 16. Conclusões da EDA

Ao finalizar a análise, registrar manualmente:

- período efetivamente coberto;
- variáveis com maior quantidade de missing;
- possíveis mudanças de regime;
- séries com comportamento tendencial;
- períodos de maior estresse/volatilidade;
- correlações relevantes;
- problemas de qualidade encontrados;
- decisões que precisam ser tomadas antes do Machine Learning.

**Importante:** nenhuma conclusão de causalidade deve ser feita apenas com esta análise exploratória.